# Week 5: Community Detection — Assignment

**Learning objectives** — In this assignment you will:

- Implement modularity from scratch using the null-model formula
- Build the Girvan-Newman edge-removal algorithm from scratch
- Compute Normalized Mutual Information (NMI) between partitions from scratch
- Sweep the Louvain resolution parameter and track quality metrics
- Quantify the stability of a stochastic community detection algorithm

## Grading

| Section | Function | Points |
|---------|----------|--------|
| 1 | `compute_modularity(G, partition)` | 20 |
| 2 | `girvan_newman(G, k)` | 25 |
| 3 | `nmi_score(partition_a, partition_b)` | 15 |
| 4 | `resolution_sweep(G, resolutions)` | 10 |
| 5 | `community_stability(G, n_runs)` | 15 |
| — | Written Questions | 15 |
| | **Total** | **100** |

## Before You Start

This assignment builds on the Week 5 lab. Make sure you are comfortable with:

- **Community definition** — dense connections within, sparse connections between groups (Lab Section 2)
- **Modularity (Q)** — compares actual edge density within communities to a random null model; Q ≈ 0 is random, Q > 0.3 is meaningful (Lab Section 3)
- **Louvain algorithm** — greedy modularity maximization with local moves + aggregation (Lab Section 4)
- **Girvan-Newman** — top-down approach: iteratively remove highest-betweenness edges to split the graph (Lab Section 5)
- **NMI** — Normalized Mutual Information measures agreement between two partitions; 1.0 = perfect match, 0.0 = independent (Lab Section 7)
- **Resolution parameter** — controls community granularity; higher resolution → more, smaller communities (Lab Section 10)

### Implementation constraints

The following functions are **banned** from your solutions:

| Banned function | Sections |
|----------------|----------|
| `nx.community.modularity` | 1, 4 |
| `nx.community.girvan_newman` | 2 |
| `sklearn.metrics.normalized_mutual_info_score`, `sklearn.metrics.mutual_info_score` | 3, 5 |

You **may** use: `G.neighbors()`, `G.nodes()`, `G.edges()`, `G.degree()`, `G.number_of_edges()`, `G.number_of_nodes()`, `nx.Graph()`, `nx.edge_betweenness_centrality()`, `nx.connected_components()`, `nx.community.louvain_communities()`, `nx.average_clustering()`, `np.log()`, `collections.Counter`.

**Important**: Later sections build on earlier ones. You must **reuse your own implementations**:
- Section 4 must use your `compute_modularity` from Section 1
- Section 5 must use your `nmi_score` from Section 3

In [1]:
import networkx as nx
import numpy as np
from netsci.loaders import load_graph
from netsci.utils import SEED

In [2]:
G_karate = load_graph("karate")
G_football = load_graph("football")

# Ground truth for karate (2 factions)
gt_karate = [
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Mr. Hi"},
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Officer"},
]

# Ground truth for football (12 conferences)
_conf_map = {}
for n in G_football.nodes():
    c = G_football.nodes[n]["conference"]
    _conf_map.setdefault(c, set()).add(n)
gt_football = list(_conf_map.values())

karate: 34 nodes, 78 edges (undirected)
football: 115 nodes, 613 edges (undirected)


---
## Section 1: Modularity from Scratch (20 pts)

Implement the **unweighted** modularity formula:

$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where:
- $A_{ij} \in \{0, 1\}$ is the unweighted adjacency matrix entry (ignore edge weights)
- $k_i$ is the degree of node $i$
- $m$ is the total number of edges
- $\delta(c_i, c_j) = 1$ if nodes $i$ and $j$ are in the same community

**Equivalent per-community form** (easier to implement):

$$Q = \sum_{c} \left[ \frac{L_c}{m} - \left( \frac{d_c}{2m} \right)^2 \right]$$

where $L_c$ is the number of internal edges in community $c$ and $d_c = \sum_{i \in c} k_i$.

**Do not use** `nx.community.modularity`.

In [15]:
def compute_modularity(G, partition):
    """Compute unweighted modularity of a partition from scratch.

    Parameters
    ----------
    G : nx.Graph
    partition : list of sets
        Each set contains the nodes in one community.

    Returns
    -------
    float
    """
    m = G.number_of_edges()
    if m == 0:
        return 0.0

    Q = 0.0

    for community in partition:
        subgraph = G.subgraph(community)
        L_c = subgraph.number_of_edges()
        d_c = sum(dict(G.degree(community)).values())

        Q += (L_c / m) - (d_c / (2 * m)) ** 2

    return float(Q)

# Am implementat formula modularitatii calculand pentru fiecare comunitate numarul de muchii interne si suma gradelor 
# Asta pentru a evalua cat de bine sunt separate comunitatile

In [16]:
# --- Validation ---
# Note: we use weight=None because the formula above is unweighted
_Q = compute_modularity(G_karate, gt_karate)
_Q_nx = nx.community.modularity(G_karate, gt_karate, weight=None)
assert abs(_Q - _Q_nx) < 1e-6, f"Got {_Q}, expected {_Q_nx}"
print(f"Karate GT modularity: {_Q:.6f} (expected: {_Q_nx:.6f})")

# Test with Louvain output on football
_louv = list(nx.community.louvain_communities(G_football, seed=SEED))
_Q2 = compute_modularity(G_football, _louv)
_Q2_nx = nx.community.modularity(G_football, _louv, weight=None)
assert abs(_Q2 - _Q2_nx) < 1e-6, f"Football: got {_Q2}, expected {_Q2_nx}"
print(f"Football Louvain modularity: {_Q2:.6f}")

# Test edge case: singleton partition (each node is its own community)
_single = [{n} for n in G_karate.nodes()]
_Q3 = compute_modularity(G_karate, _single)
_Q3_nx = nx.community.modularity(G_karate, _single, weight=None)
assert abs(_Q3 - _Q3_nx) < 1e-6, f"Singleton: got {_Q3}, expected {_Q3_nx}"
print(f"Singleton partition modularity: {_Q3:.6f} (should be negative)")

# Test: one giant community (all nodes together)
_all_one = [set(G_karate.nodes())]
_Q4 = compute_modularity(G_karate, _all_one)
assert abs(_Q4 - 0.0) < 1e-6, f"Single community should give Q=0, got {_Q4}"
print(f"Single community modularity: {_Q4:.6f} (should be 0)")
print("Section 1 passed!")

Karate GT modularity: 0.358235 (expected: 0.358235)
Football Louvain modularity: 0.604570
Singleton partition modularity: -0.049803 (should be negative)
Single community modularity: 0.000000 (should be 0)
Section 1 passed!


---
## Section 2: Girvan-Newman from Scratch (25 pts)

Implement the Girvan-Newman algorithm **from scratch**. **Do NOT** call `nx.community.girvan_newman`.

The algorithm:

1. **Start** with a copy of the input graph
2. **Repeat** until the graph has at least `k` connected components:
   - Compute edge betweenness centrality for all edges (use `nx.edge_betweenness_centrality`)
   - Remove the edge with the **highest** betweenness centrality
   - If multiple edges tie for highest betweenness, remove any one of them
3. **Return** the connected components as a list of sets

**Implementation details:**
- Work on a **copy** of the graph — do not modify the original
- After removing edges, use `nx.connected_components()` to count components
- Return the partition as a list of sets (each set = one community), sorted by size descending

In [17]:
def girvan_newman(G, k=2):
    """Split a graph into k communities using Girvan-Newman.

    Do NOT call nx.community.girvan_newman.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    k : int
        Target number of communities.

    Returns
    -------
    list of sets — partition into k communities, sorted by size descending
    """
    import networkx as nx

    H = G.copy()

    while nx.number_connected_components(H) < k and H.number_of_edges() > 0:
        betweenness = nx.edge_betweenness_centrality(H)
        edge_to_remove = max(betweenness, key=betweenness.get)
        H.remove_edge(*edge_to_remove)

    communities = list(nx.connected_components(H))
    communities.sort(key=len, reverse=True)
    return communities

# Am implementat algoritmul Girvan-Newman prin eliminarea iterativa a muchiilor cu betweenness maxim
# Asta pana cand graful se separa in k comunitati
# betweenness maxim - muchia (sau nodul) care apare cel mai des pe cele mai scurte drumuri dintre perechi de noduri

In [18]:
# --- Validation ---
# Karate club: split into 2
_gn2 = girvan_newman(G_karate, k=2)
assert isinstance(_gn2, list)
assert len(_gn2) == 2, f"Expected 2 communities, got {len(_gn2)}"
assert all(isinstance(c, set) for c in _gn2)
_all_nodes_gn = set()
for c in _gn2:
    _all_nodes_gn |= c
assert _all_nodes_gn == set(G_karate.nodes()), "All nodes must be assigned"

# Should produce a decent partition
_Q_gn = nx.community.modularity(G_karate, _gn2, weight=None)
assert _Q_gn > 0.3, f"GN modularity {_Q_gn:.3f} too low for karate (expected > 0.3)"
print(f"GN(k=2) on karate: sizes={sorted([len(c) for c in _gn2], reverse=True)}, Q={_Q_gn:.4f}")

# Sorted by size descending
assert len(_gn2[0]) >= len(_gn2[1]), "Communities should be sorted by size descending"

# Original graph unchanged
assert G_karate.number_of_edges() == 78, "Original graph must not be modified"

# Karate club: split into 4
_gn4 = girvan_newman(G_karate, k=4)
assert len(_gn4) == 4, f"Expected 4 communities, got {len(_gn4)}"
_all4 = set()
for c in _gn4:
    _all4 |= c
assert _all4 == set(G_karate.nodes())
_Q_gn4 = nx.community.modularity(G_karate, _gn4, weight=None)
print(f"GN(k=4) on karate: sizes={sorted([len(c) for c in _gn4], reverse=True)}, Q={_Q_gn4:.4f}")

# Football: split into 12 conferences
_gn_fb = girvan_newman(G_football, k=12)
assert len(_gn_fb) == 12, f"Expected 12 communities, got {len(_gn_fb)}"
_Q_gn_fb = nx.community.modularity(G_football, _gn_fb, weight=None)
assert _Q_gn_fb > 0.4, f"GN modularity on football {_Q_gn_fb:.3f} too low"
print(f"GN(k=12) on football: Q={_Q_gn_fb:.4f}")
print("Section 2 passed!")

GN(k=2) on karate: sizes=[19, 15], Q=0.3600
GN(k=4) on karate: sizes=[18, 10, 5, 1], Q=0.3632
GN(k=12) on football: Q=0.5973
Section 2 passed!


---
## Section 3: Normalized Mutual Information (15 pts)

Implement NMI from scratch. **Do NOT** use `sklearn.metrics.normalized_mutual_info_score` or any library NMI function.

NMI measures the agreement between two partitions on a scale of 0 (independent) to 1 (identical).

$$\text{NMI}(U, V) = \frac{2 \cdot I(U; V)}{H(U) + H(V)}$$

where:
- $H(U) = -\sum_i \frac{|U_i|}{N} \ln \frac{|U_i|}{N}$ is the entropy of partition $U$
- $I(U; V) = \sum_i \sum_j \frac{|U_i \cap V_j|}{N} \ln \frac{N \cdot |U_i \cap V_j|}{|U_i| \cdot |V_j|}$ is the mutual information
- Skip terms where $|U_i \cap V_j| = 0$ (since $0 \cdot \ln 0 = 0$)
- If $H(U) + H(V) = 0$ (both partitions place all nodes in one group), return 1.0

In [25]:
def nmi_score(partition_a, partition_b):
    """Compute Normalized Mutual Information between two partitions."""
    import numpy as np

    # All nodes
    nodes = set().union(*partition_a)
    N = len(nodes)

    # Convert partitions to label dicts
    def to_labels(partition):
        label_map = {}
        for i, comm in enumerate(partition):
            for n in comm:
                label_map[n] = i
        return label_map

    labels_a = to_labels(partition_a)
    labels_b = to_labels(partition_b)

    # Build contingency matrix
    comms_a = list(partition_a)
    comms_b = list(partition_b)

    I = 0.0
    H_a = 0.0
    H_b = 0.0

    # Compute H(U)
    for ca in comms_a:
        p = len(ca) / N
        if p > 0:
            H_a -= p * np.log(p)

    # Compute H(V)
    for cb in comms_b:
        p = len(cb) / N
        if p > 0:
            H_b -= p * np.log(p)

    # Compute I(U, V)
    for ca in comms_a:
        for cb in comms_b:
            inter = len(ca & cb)
            if inter == 0:
                continue
            p_uv = inter / N
            p_u = len(ca) / N
            p_v = len(cb) / N
            I += p_uv * np.log(p_uv / (p_u * p_v))

    # Handle edge case
    if H_a + H_b == 0:
        return 1.0

    return float(2 * I / (H_a + H_b))

# Am calculat NMI pentru a masura cat de similare sunt doua partitii, am folosit entropia si informatia mutuala

In [26]:
# --- Validation ---
from sklearn.metrics import normalized_mutual_info_score as _sklearn_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

# Perfect match should be 1.0
_nmi_perfect = nmi_score(gt_karate, gt_karate)
assert abs(_nmi_perfect - 1.0) < 1e-6, f"Perfect NMI should be 1.0, got {_nmi_perfect}"
print(f"NMI(GT, GT) = {_nmi_perfect:.4f}")

# All-in-one vs all-in-one should be 1.0 (edge case)
_all_one_a = [set(G_karate.nodes())]
_all_one_b = [set(G_karate.nodes())]
_nmi_trivial = nmi_score(_all_one_a, _all_one_b)
assert abs(_nmi_trivial - 1.0) < 1e-6, f"Trivial NMI should be 1.0, got {_nmi_trivial}"

# Louvain vs ground truth on karate
_louv_k = list(nx.community.louvain_communities(G_karate, seed=SEED))
_nmi_louv = nmi_score(_louv_k, gt_karate)
_nodes_k = list(G_karate.nodes())
_nmi_sklearn = _sklearn_nmi(
    _to_labels(_louv_k, _nodes_k), _to_labels(gt_karate, _nodes_k)
)
assert abs(_nmi_louv - _nmi_sklearn) < 0.05, (
    f"NMI(Louvain, GT) = {_nmi_louv:.4f}, sklearn says {_nmi_sklearn:.4f}"
)
print(f"NMI(Louvain, GT) on karate = {_nmi_louv:.4f} (sklearn: {_nmi_sklearn:.4f})")

# Football: Louvain vs conference ground truth
_louv_fb = list(nx.community.louvain_communities(G_football, seed=SEED))
_nmi_fb = nmi_score(_louv_fb, gt_football)
_nodes_fb = list(G_football.nodes())
_nmi_fb_sk = _sklearn_nmi(
    _to_labels(_louv_fb, _nodes_fb), _to_labels(gt_football, _nodes_fb)
)
assert abs(_nmi_fb - _nmi_fb_sk) < 0.05, (
    f"Football NMI = {_nmi_fb:.4f}, sklearn = {_nmi_fb_sk:.4f}"
)
print(f"NMI(Louvain, GT) on football = {_nmi_fb:.4f}")

# NMI should be > 0 but < 1 for imperfect match
assert 0 < _nmi_louv < 1.0, f"Expected 0 < NMI < 1, got {_nmi_louv}"
print("Section 3 passed!")

NMI(GT, GT) = 1.0000
NMI(Louvain, GT) on karate = 0.5942 (sklearn: 0.5942)
NMI(Louvain, GT) on football = 0.8903
Section 3 passed!


---
## Section 4: Resolution Sweep (10 pts)

Sweep over a list of Louvain resolution values and collect quality metrics at each resolution.

**Implementation requirements:**
- **Reuse your own** `compute_modularity` from Section 1 (do NOT call `nx.community.modularity`)
- Use `nx.community.louvain_communities(G, resolution=r, seed=SEED)` for detection
- Return a dictionary with keys:
  - `"resolution"` — list of resolution values
  - `"modularity"` — list of modularity values (from YOUR `compute_modularity`)
  - `"n_communities"` — list of community counts
  - `"best_resolution"` — the resolution that achieved the highest modularity

In [21]:
def resolution_sweep(G, resolutions):
    """Sweep Louvain resolution and collect quality metrics.

    Reuse your compute_modularity() from Section 1.
    Do NOT call nx.community.modularity.
    """
    modularities = []
    n_communities = []

    for r in resolutions:
        partition = list(nx.community.louvain_communities(G, resolution=r, seed=SEED))
        q = compute_modularity(G, partition)

        modularities.append(q)
        n_communities.append(len(partition))

    best_idx = modularities.index(max(modularities))

    return {
        "resolution": list(resolutions),
        "modularity": modularities,
        "n_communities": n_communities,
        "best_resolution": resolutions[best_idx],
    }

# Am testat mai multe valori ale rezolutiei Louvain si am calculat modularitatea si numarul de comunitati pentru fiecare si am ales rezolutia optima

In [22]:
# --- Validation ---
_res = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
_result = resolution_sweep(G_football, _res)

assert set(_result.keys()) == {"resolution", "modularity", "n_communities", "best_resolution"}
assert len(_result["resolution"]) == len(_res)
assert len(_result["modularity"]) == len(_res)
assert len(_result["n_communities"]) == len(_res)

# More communities at higher resolution
assert _result["n_communities"][-1] > _result["n_communities"][0], (
    "Higher resolution should produce more communities"
)

# Verify modularity values against nx (unweighted)
for i, r in enumerate(_res):
    _p = list(nx.community.louvain_communities(G_football, resolution=r, seed=SEED))
    _q_expected = nx.community.modularity(G_football, _p, weight=None)
    assert abs(_result["modularity"][i] - _q_expected) < 1e-6, (
        f"Modularity mismatch at r={r}: got {_result['modularity'][i]}, expected {_q_expected}"
    )

# best_resolution should be in the input list
assert _result["best_resolution"] in _res, (
    f"best_resolution {_result['best_resolution']} not in input resolutions"
)

# best_resolution should correspond to max modularity
_best_idx = _result["modularity"].index(max(_result["modularity"]))
assert abs(_result["best_resolution"] - _res[_best_idx]) < 1e-9, (
    f"best_resolution should match the resolution with highest Q"
)

print("Resolution | Communities | Modularity")
print("-" * 42)
for i in range(len(_res)):
    marker = " <-- best" if abs(_res[i] - _result["best_resolution"]) < 1e-9 else ""
    print(f"    {_res[i]:.1f}    |     {_result['n_communities'][i]:2d}      | {_result['modularity'][i]:.4f}{marker}")
print(f"\nBest resolution: {_result['best_resolution']}")
print("Section 4 passed!")

Resolution | Communities | Modularity
------------------------------------------
    0.5    |      6      | 0.5828
    0.8    |      7      | 0.6006
    1.0    |     10      | 0.6046 <-- best
    1.2    |     10      | 0.6044
    1.5    |     12      | 0.6005
    2.0    |     12      | 0.6005
    3.0    |     12      | 0.6005

Best resolution: 1.0
Section 4 passed!


---
## Section 5: Community Stability (15 pts)

Louvain is stochastic — different random seeds produce different partitions.
Measure how stable the algorithm is by running it multiple times and computing the
average pairwise NMI between all pairs of resulting partitions.

**Implementation requirements:**
- Run `nx.community.louvain_communities(G, seed=i)` for `i = 0, 1, ..., n_runs - 1`
- **Reuse your own** `nmi_score` from Section 3 (do NOT use sklearn)
- Compute NMI for every pair `(i, j)` where `i < j`
- Return a dictionary with:
  - `"mean_nmi"` — average pairwise NMI (float)
  - `"min_nmi"` — minimum pairwise NMI (float)
  - `"max_nmi"` — maximum pairwise NMI (float)
  - `"n_unique"` — number of distinct community counts observed across runs (int)

In [23]:
def community_stability(G, n_runs=10):
    """Measure stability of Louvain across multiple runs.

    Reuse your nmi_score() from Section 3.
    Do NOT use sklearn.metrics.normalized_mutual_info_score.
    """
    import numpy as np

    partitions = [
        list(nx.community.louvain_communities(G, seed=i))
        for i in range(n_runs)
    ]

    nmis = []
    community_counts = []

    for p in partitions:
        community_counts.append(len(p))

    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            nmis.append(nmi_score(partitions[i], partitions[j]))

    return {
        "mean_nmi": float(np.mean(nmis)) if nmis else 1.0,
        "min_nmi": float(np.min(nmis)) if nmis else 1.0,
        "max_nmi": float(np.max(nmis)) if nmis else 1.0,
        "n_unique": len(set(community_counts)),
    }

# Am rulat algoritmul Louvain de mai multe ori si am calculat NMI intre toate perechile de rezultate pentru a evalua stabilitatea comunitatilor

In [24]:
# --- Validation ---
_stab_k = community_stability(G_karate, n_runs=10)
assert set(_stab_k.keys()) == {"mean_nmi", "min_nmi", "max_nmi", "n_unique"}
assert 0 <= _stab_k["min_nmi"] <= _stab_k["mean_nmi"] <= _stab_k["max_nmi"] <= 1.0
assert isinstance(_stab_k["n_unique"], int) and _stab_k["n_unique"] >= 1

print(f"Karate stability (10 runs):")
print(f"  Mean NMI: {_stab_k['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_k['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_k['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_k['n_unique']}")

_stab_fb = community_stability(G_football, n_runs=10)
assert 0 <= _stab_fb["min_nmi"] <= _stab_fb["mean_nmi"] <= _stab_fb["max_nmi"] <= 1.0

# Football should be reasonably stable (clear community structure)
assert _stab_fb["mean_nmi"] > 0.85, (
    f"Football mean NMI {_stab_fb['mean_nmi']:.3f} too low — expected > 0.85"
)
print(f"\nFootball stability (10 runs):")
print(f"  Mean NMI: {_stab_fb['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_fb['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_fb['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_fb['n_unique']}")

# Verify against sklearn
from sklearn.metrics import normalized_mutual_info_score as _sk_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

_parts = [list(nx.community.louvain_communities(G_football, seed=i)) for i in range(10)]
_nodes_fb = list(G_football.nodes())
_sk_nmis = []
for i in range(10):
    for j in range(i + 1, 10):
        _sk_nmis.append(_sk_nmi(
            _to_labels(_parts[i], _nodes_fb),
            _to_labels(_parts[j], _nodes_fb)
        ))
_sk_mean = np.mean(_sk_nmis)
assert abs(_stab_fb["mean_nmi"] - _sk_mean) < 0.05, (
    f"Mean NMI {_stab_fb['mean_nmi']:.4f} differs from sklearn {_sk_mean:.4f}"
)
print(f"  (sklearn verification: mean={_sk_mean:.4f})")
print("Section 5 passed!")

Karate stability (10 runs):
  Mean NMI: 0.9214
  Min NMI:  0.6880
  Max NMI:  1.0000
  Unique community counts: 2

Football stability (10 runs):
  Mean NMI: 0.9629
  Min NMI:  0.9198
  Max NMI:  1.0000
  Unique community counts: 2
  (sklearn verification: mean=0.9629)
Section 5 passed!


---
## Written Questions (15 pts)

### Question 1 (5 pts)

Run Girvan-Newman and Louvain on the karate club with `k=2` / default resolution and compare:

(a) Report the community sizes and modularity Q for each algorithm. Are the partitions identical? If not, which nodes are assigned differently?

(b) Girvan-Newman is $O(m^2 n)$ while Louvain is nearly $O(n \log n)$. Given their quality and speed trade-off, when would you choose Girvan-Newman over Louvain in practice?

(c) Compute the NMI between the two partitions (using your `nmi_score`). Is the disagreement large or small? What does this tell you about the "uniqueness" of community structure in this network?

**You must include numerical values from your code.** Show the code cell you used to compute them.

*Hints to guide your thinking:*
- *Girvan-Newman is deterministic (edge betweenness gives a unique ranking), while Louvain is stochastic. What are the implications?*
- *Think about "bridge" nodes between communities — are they the ones that differ?*
- *Consider the trade-off: Girvan-Newman gives a full dendrogram (hierarchical decomposition), while Louvain gives a flat partition at one level.*

**Your Answer:**
Girvan-Newman si Louvain produc de obicei partitii foarte asemanatoare pe karate club, cu valori apropiate ale modularitatii. Daca partitiile nu sunt identice, diferentele apar de regula la nodurile de frontiera dintre comunitati. Louvain este mai potrivit in practica deoarece este mult mai rapid si ofera rezultate foarte bune, in timp ce Girvan-Newman este util mai ales pe grafuri mici sau cand vrem o separare ierarhica. Daca NMI este aproape de 1, atunci dezacordul dintre partitii este mic si structura comunitatilor este clara.

Code:
# Girvan-Newman vs Louvain pe karate club

# 1. Ruleaza algoritmii
gn_part = girvan_newman(G_karate, k=2)
louv_part = list(nx.community.louvain_communities(G_karate, seed=SEED))

# 2. Marimile comunitatilor
gn_sizes = sorted([len(c) for c in gn_part], reverse=True)
louv_sizes = sorted([len(c) for c in louv_part], reverse=True)

# 3. Modularity Q
gn_q = compute_modularity(G_karate, gn_part)
louv_q = compute_modularity(G_karate, louv_part)

# 4. Verifica daca partitiile sunt identice
def same_partition(part_a, part_b):
    a = {frozenset(c) for c in part_a}
    b = {frozenset(c) for c in part_b}
    return a == b

identice = same_partition(gn_part, louv_part)

# 5. Noduri asignate diferit
def node_to_comm(partition):
    m = {}
    for i, comm in enumerate(partition):
        for node in comm:
            m[node] = i
    return m

gn_map = node_to_comm(gn_part)
louv_map = node_to_comm(louv_part)

different_nodes = []
for node in G_karate.nodes():
    same_gn = gn_part[gn_map[node]]
    same_louv = louv_part[louv_map[node]]
    if same_gn != same_louv:
        different_nodes.append(node)

# 6. NMI
nmi_val = nmi_score(gn_part, louv_part)

# 7. Afisare rezultate
print("Girvan-Newman:")
print("  marimi comunitati =", gn_sizes)
print("  modularity Q =", round(gn_q, 4))

print("\nLouvain:")
print("  marimi comunitati =", louv_sizes)
print("  modularity Q =", round(louv_q, 4))

print("\nPartitii identice?", identice)
print("Noduri asignate diferit:", sorted(different_nodes))
print("NMI =", round(nmi_val, 4))


### Question 2 (5 pts)

Fortunato & Barthélemy (2007) proved that modularity optimization has a **resolution limit**: it may fail to detect communities smaller than $\sim\sqrt{2m}$ nodes, where $m$ is the total number of edges.

(a) For the football network, compute $\sqrt{2m}$. How many of the 12 ground-truth conferences have fewer members than this threshold?

(b) Yet Louvain still finds ~10 communities with Q > 0.55 on this network. Explain why the resolution limit doesn't completely destroy performance here. (*Hint: the worst case for the resolution limit involves specific graph topologies.*)

(c) **Use your Section 4 results**: At which resolution does the number of detected communities best match the 12 ground-truth conferences? Is this the same resolution that maximizes modularity? What does this discrepancy (if any) tell you about using modularity as the sole quality criterion?

*Hints to guide your thinking:*
- *The resolution limit is a worst-case result for two cliques connected by a single edge. Real communities have denser internal connections and sparser inter-community links.*
- *Compare the resolution that gives ~12 communities vs the one that maximizes Q. If they differ, it means maximizing Q doesn't recover the "right" number of communities.*
- *This is why the resolution parameter exists — it lets you override Q-maximization and zoom in/out.*

**Your Answer:**
(a) Pragul este √(2m) = …, iar numarul de conferinte cu mai putine noduri decat acest prag este … din 12.

(b) Limita de rezolutie este un caz teoretic „worst-case”. In reteaua football, comunitatile sunt totusi bine separate, cu multe muchii interne si putine intre conferinte, asa ca Louvain le poate detecta destul de bine chiar daca unele sunt sub prag.

(c) Din rezultatele din Section 4:

rezolutia care da 12 comunitati este 1.5 (la fel si 2.0 si 3.0)
rezolutia care maximizeaza modularitatea este 1.0, cu Q = 0.6046

Deci nu este aceeasi rezolutie. Asta arata ca maximizarea modularitatii nu recupereaza neaparat numarul „corect” de comunitati. De aceea, parametrul de rezolutie este util pentru a controla cat de fina sau grosiera este impartirea.


### Question 3 (5 pts)

Your Section 5 measures the stability of Louvain across runs.

(a) Report the mean, min, and max pairwise NMI for both karate and football. Which network has more stable community assignments? Why? (*Think about the structure of each network.*)

(b) Identify the "unstable" nodes — nodes most likely to switch communities across runs. What structural property do these nodes share? (*Hint: think about their position relative to community boundaries.*)

(c) If you needed to produce a single "consensus" partition from 100 Louvain runs, describe an algorithm to do so. How would you use NMI or modularity to select or construct the best result?

*Hints to guide your thinking:*
- *Small networks have fewer constraints, so the algorithm has more "freedom" to assign ambiguous nodes differently.*
- *Bridge nodes, nodes with low clustering, or nodes at the boundary of communities are structurally ambiguous.*
- *For consensus: consider building a co-occurrence matrix (how often each pair of nodes ends up in the same community) and clustering that. Alternatively, pick the run whose partition has the highest average NMI with all other runs.*

**Your Answer:**
(a) Din rezultate:

Karate: mean NMI = 0.9214, min NMI = 0.6880, max NMI = 1.0000
Football: mean NMI = 0.9629, min NMI = 0.9198, max NMI = 1.0000

Reteaua football este mai stabila, deoarece are valori NMI mai mari si variatie mai mica intre rulari, deci comunitatile sunt mai clar separate.

(b) Nodurile „instabile” sunt cele aflate la granita dintre comunitati, adica noduri-punte, cu legaturi spre mai multe grupuri. Acestea pot fi mutate mai usor intre comunitati de la o rulare la alta.

(c) Pentru o partitie de tip „consensus”, putem construi o matrice de co-occurenta care arata cat de des ajung doua noduri in aceeasi comunitate si apoi clusterizam aceasta matrice. O varianta mai simpla este sa alegem rularea care are cel mai mare NMI mediu fata de toate celelalte rulari. Astfel obtinem partitia cea mai reprezentativa si mai stabila.
